# Ensemble Evaluation - F1 Podium Prediction

Evaluasi lanjutan: Walk-Forward Validation, Probability Calibration, Ensemble/Blending.

## Agenda:
1. Walk-Forward Validation (4 folds: 2019-2022)
2. Probability Calibration (Platt scaling + Isotonic)
3. Ensemble/Blending (Classifier + Ranker + Baseline)
4. Final Test Evaluation (2024-2025)

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

print('Library loaded.')

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_parquet('../data/processed/model_dataset.parquet')
print(f'Dataset shape: {df.shape}')
print(f'Years: {df["year"].min()} - {df["year"].max()}')
print(f'Unique races: {df["raceId"].nunique()}')

In [ ]:
# Relevance score
df['relevance'] = df['positionOrder'].apply(
    lambda x: 3 if x == 1 else (2 if x == 2 else (1 if x == 3 else 0))
)

# Feature columns
id_columns = ['raceId', 'driverId', 'constructorId', 'year', 'round',
              'circuitId', 'date', 'race_name', 'driverRef', 'team',
              'positionOrder', 'is_podium', 'relevance']

feature_cols = [c for c in df.columns if c not in id_columns]
print(f'Jumlah fitur: {len(feature_cols)}')

## 2. Walk-Forward Validation (Section 9.3)

Skema:
- Fold 1: Train 2014-2018, Validate 2019
- Fold 2: Train 2014-2019, Validate 2020
- Fold 3: Train 2014-2020, Validate 2021
- Fold 4: Train 2014-2021, Validate 2022

Tiap fold: train classifier + ranker, evaluasi metrik.

In [ ]:
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier, LGBMRanker
import lightgbm as lgb

def prepare_fold_data(df, train_end_year, val_year):
    """Siapkan data untuk satu fold walk-forward validation."""
    train_mask = df['year'] <= train_end_year
    val_mask = df['year'] == val_year
    
    X_train = df[train_mask][feature_cols].copy()
    y_train = df[train_mask]['is_podium'].copy()
    rel_train = df[train_mask]['relevance'].copy()
    
    X_val = df[val_mask][feature_cols].copy()
    y_val = df[val_mask]['is_podium'].copy()
    rel_val = df[val_mask]['relevance'].copy()
    
    val_meta = df[val_mask][['raceId', 'year', 'round', 'driverRef', 'team',
                              'positionOrder', 'is_podium', 'relevance']].copy()
    
    # Impute median
    median_vals = X_train.median()
    X_train = X_train.fillna(median_vals)
    X_val = X_val.fillna(median_vals)
    
    # Group sizes
    group_train = val_meta[val_meta.index.isin(X_train.index[:0])].groupby('raceId').size().values
    # Better: get group from df
    train_group = df[train_mask].groupby('raceId').size().values
    val_group = df[val_mask].groupby('raceId').size().values
    
    return X_train, y_train, rel_train, train_group, X_val, y_val, rel_val, val_group, val_meta


def run_single_fold(X_train, y_train, rel_train, group_train, 
                    X_val, y_val, rel_val, group_val, val_meta):
    """Train dan evaluasi satu fold."""
    # Classifier
    clf = LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
    )
    clf.fit(X_train, y_train)
    clf_prob = clf.predict_proba(X_val)[:, 1]
    
    # Ranker
    ranker = LGBMRanker(
        objective='lambdarank', n_estimators=200, max_depth=6,
        learning_rate=0.1, num_leaves=31, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
        random_state=42, n_jobs=-1, verbose=-1
    )
    ranker.fit(
        X_train, rel_train, group=group_train,
        eval_set=[(X_val, rel_val)], eval_group=[group_val],
        eval_at=[3], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
    )
    ranker_score = ranker.predict(X_val)
    
    # Evaluasi
    results = {}
    for name, scores in [('Classifier', clf_prob), ('Ranker', ranker_score)]:
        meta = val_meta.copy()
        meta['score'] = scores
        
        hits_total, exact_total, win_total, n_races = 0, 0, 0, 0
        
        for race_id, group in meta.groupby('raceId'):
            if len(group) < 3:
                continue
            actual_podium = group[group['positionOrder'] <= 3]['driverRef'].tolist()
            actual_p1 = group[group['positionOrder'] == 1]['driverRef'].values
            
            top3 = group.nlargest(3, 'score')
            predicted_podium = top3['driverRef'].tolist()
            predicted_p1 = top3.iloc[0]['driverRef']
            
            hits_total += sum(1 for d in actual_podium if d in predicted_podium)
            exact_total += int(all(d in predicted_podium for d in actual_podium))
            win_total += int(actual_p1[0] == predicted_p1)
            n_races += 1
        
        results[name] = {
            'podium_hit_rate': hits_total / (n_races * 3),
            'winner_accuracy': win_total / n_races,
            'exact_drivers': exact_total / n_races,
            'n_races': n_races
        }
    
    return results, clf, ranker

In [ ]:
# Jalankan walk-forward validation
folds = [
    (2018, 2019, 'Fold 1: 2014-2018 -> 2019'),
    (2019, 2020, 'Fold 2: 2014-2019 -> 2020'),
    (2020, 2021, 'Fold 3: 2014-2020 -> 2021'),
    (2021, 2022, 'Fold 4: 2014-2021 -> 2022'),
]

wf_results = []

for train_end, val_year, label in folds:
    print(f'\n=== {label} ===')
    
    train_mask = df['year'] <= train_end
    val_mask = df['year'] == val_year
    
    X_tr = df[train_mask][feature_cols].copy()
    y_tr = df[train_mask]['is_podium'].copy()
    rel_tr = df[train_mask]['relevance'].copy()
    
    X_va = df[val_mask][feature_cols].copy()
    y_va = df[val_mask]['is_podium'].copy()
    rel_va = df[val_mask]['relevance'].copy()
    va_meta = df[val_mask][['raceId', 'year', 'driverRef', 'positionOrder', 'is_podium', 'relevance']].copy()
    
    # Impute
    median_vals = X_tr.median()
    X_tr = X_tr.fillna(median_vals)
    X_va = X_va.fillna(median_vals)
    
    # Group sizes
    tr_group = df[train_mask].groupby('raceId').size().values
    va_group = df[val_mask].groupby('raceId').size().values
    
    results, _, _ = run_single_fold(X_tr, y_tr, rel_tr, tr_group,
                                     X_va, y_va, rel_va, va_group, va_meta)
    
    for model_name, metrics in results.items():
        print(f'  {model_name}: PHR={metrics["podium_hit_rate"]:.4f}, '
              f'Winner={metrics["winner_accuracy"]:.4f}, '
              f'Exact={metrics["exact_drivers"]:.4f}')
        wf_results.append({
            'fold': label,
            'val_year': val_year,
            'model': model_name,
            **metrics
        })

wf_df = pd.DataFrame(wf_results)
print('\n=== WALK-FORWARD SUMMARY ===')
display(wf_df)

In [ ]:
# Visualisasi walk-forward results
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

plt.figure(figsize=(12, 5))

for i, model in enumerate(['Classifier', 'Ranker']):
    subset = wf_df[wf_df['model'] == model]
    plt.plot(subset['val_year'], subset['podium_hit_rate'], 
             marker='o', label=f'{model} - Podium Hit Rate', linewidth=2)
    plt.plot(subset['val_year'], subset['winner_accuracy'], 
             marker='s', linestyle='--', label=f'{model} - Winner Accuracy', linewidth=2)

plt.xlabel('Validation Year')
plt.ylabel('Score')
plt.title('Walk-Forward Validation: Performance per Season')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/walk_forward_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/walk_forward_validation.png')

## 3. Probability Calibration (Section 14.4)

Kalibrasi probabilitas classifier menggunakan Platt scaling dan Isotonic regression.

In [ ]:
# Siapkan data train (2014-2022) dan validation (2023) untuk kalibrasi
train_mask = df['year'] <= 2022
val_mask = df['year'] == 2023
test_mask = df['year'] >= 2024

X_train = df[train_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_train = df[train_mask]['is_podium']

X_val = df[val_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_val = df[val_mask]['is_podium']

X_test = df[test_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_test = df[test_mask]['is_podium']

test_meta = df[test_mask][['raceId', 'year', 'round', 'driverRef', 'team',
                            'positionOrder', 'is_podium']].copy()

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# Train LightGBM classifier
lgb_calib = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
)
lgb_calib.fit(X_train, y_train)

# Probabilitas sebelum kalibrasi
prob_val_raw = lgb_calib.predict_proba(X_val)[:, 1]
prob_test_raw = lgb_calib.predict_proba(X_test)[:, 1]

print('Model trained. Probabilities calculated.')

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Platt scaling (sigmoid)
calib_platt = CalibratedClassifierCV(lgb_calib, method='sigmoid', cv='prefit')
calib_platt.fit(X_val, y_val)
prob_test_platt = calib_platt.predict_proba(X_test)[:, 1]

# Isotonic regression
calib_iso = CalibratedClassifierCV(lgb_calib, method='isotonic', cv='prefit')
calib_iso.fit(X_val, y_val)
prob_test_iso = calib_iso.predict_proba(X_test)[:, 1]

print('Calibration completed.')

In [ ]:
from sklearn.metrics import brier_score_loss

# Bandingkan Brier Score sebelum dan sesudah kalibrasi
brier_raw = brier_score_loss(y_test, prob_test_raw)
brier_platt = brier_score_loss(y_test, prob_test_platt)
brier_iso = brier_score_loss(y_test, prob_test_iso)

print('=== BRIER SCORE COMPARISON ===')
print(f'Raw model:       {brier_raw:.4f}')
print(f'Platt scaling:   {brier_platt:.4f}')
print(f'Isotonic:        {brier_iso:.4f}')

In [ ]:
# Reliability curve visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

probs = [
    ('Raw (Uncalibrated)', prob_test_raw, 'steelblue'),
    ('Platt Scaling (Sigmoid)', prob_test_platt, 'seagreen'),
    ('Isotonic Regression', prob_test_iso, 'coral'),
]

for ax, (name, prob, color) in zip(axes, probs):
    prob_true, prob_pred = calibration_curve(y_test, prob, n_bins=10, strategy='uniform')
    
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', alpha=0.5)
    ax.plot(prob_pred, prob_true, marker='o', color=color, linewidth=2, label=name)
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(name)
    ax.legend(loc='lower right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/figures/calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/calibration_curve.png')

## 4. Ensemble / Blending (Section 12.4)

Gabungkan classifier, ranker, dan baseline score dengan bobot optimal.

Formula:
```
final_score = w1 * classifier_prob + w2 * ranker_score + w3 * baseline_score
```

In [ ]:
# Siapkan data untuk ensemble
train_mask = df['year'] <= 2022
val_mask = df['year'] == 2023
test_mask = df['year'] >= 2024

X_tr = df[train_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_tr = df[train_mask]['is_podium']
rel_tr = df[train_mask]['relevance']
tr_group = df[train_mask].groupby('raceId').size().values

X_va = df[val_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_va = df[val_mask]['is_podium']
rel_va = df[val_mask]['relevance']
va_group = df[val_mask].groupby('raceId').size().values
va_meta = df[val_mask][['raceId', 'year', 'driverRef', 'constructorId',
                         'positionOrder', 'is_podium']].copy()

X_te = df[test_mask][feature_cols].fillna(df[train_mask][feature_cols].median())
y_te = df[test_mask]['is_podium']
te_meta = df[test_mask][['raceId', 'year', 'driverRef', 'constructorId',
                          'positionOrder', 'is_podium']].copy()

# Train classifier
clf = LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    class_weight='balanced', subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1
)
clf.fit(X_tr, y_tr)

# Train ranker
ranker = LGBMRanker(
    objective='lambdarank', n_estimators=200, max_depth=6,
    learning_rate=0.1, num_leaves=31, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1
)
ranker.fit(X_tr, rel_tr, group=tr_group,
           eval_set=[(X_va, rel_va)], eval_group=[va_group],
           eval_at=[3], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

print('Models trained for ensemble.')

In [ ]:
# Prediksi scores untuk validation dan test
clf_prob_va = clf.predict_proba(X_va)[:, 1]
ranker_score_va = ranker.predict(X_va)

clf_prob_te = clf.predict_proba(X_te)[:, 1]
ranker_score_te = ranker.predict(X_te)

# Normalisasi ranker score ke [0,1] untuk blending
def normalize_scores(scores):
    mn, mx = scores.min(), scores.max()
    if mx > mn:
        return (scores - mn) / (mx - mn)
    return scores

ranker_score_va_norm = normalize_scores(ranker_score_va)
ranker_score_te_norm = normalize_scores(ranker_score_te)

In [ ]:
# Grid search bobot optimal berdasarkan validation set
def evaluate_ensemble(val_meta, clf_prob, ranker_score, w1, w2):
    """Evaluasi ensemble dengan bobot tertentu."""
    w3 = 1.0 - w1 - w2
    if w3 < 0:
        return None
    
    final_score = w1 * clf_prob + w2 * ranker_score + w3 * 0
    
    meta = val_meta.copy()
    meta['score'] = final_score
    
    hits_total, n_races = 0, 0
    for race_id, group in meta.groupby('raceId'):
        if len(group) < 3:
            continue
        actual_podium = group[group['positionOrder'] <= 3]['driverRef'].tolist()
        top3 = group.nlargest(3, 'score')
        predicted = top3['driverRef'].tolist()
        hits_total += sum(1 for d in actual_podium if d in predicted)
        n_races += 1
    
    return hits_total / (n_races * 3) if n_races > 0 else 0

# Coba kombinasi bobot
best_score = 0
best_weights = (0.0, 0.0)

for w1 in np.arange(0, 1.05, 0.1):
    for w2 in np.arange(0, 1.05 - w1, 0.1):
        phr = evaluate_ensemble(va_meta, clf_prob_va, ranker_score_va_norm, w1, w2)
        if phr and phr > best_score:
            best_score = phr
            best_weights = (w1, w2)

w1_opt, w2_opt = best_weights
w3_opt = 1.0 - w1_opt - w2_opt

print(f'Bobot optimal (berdasarkan validation PHR):')
print(f'  Classifier: {w1_opt:.1f}')
print(f'  Ranker:     {w2_opt:.1f}')
print(f'  Validation PHR: {best_score:.4f}')

## 5. Final Test Evaluation

Evaluasi semua pendekatan di test set (2024-2025).

In [ ]:
def evaluate_on_test(te_meta, scores, label='Model'):
    """Evaluasi di test set, return metrics."""
    meta = te_meta.copy()
    meta['score'] = scores
    
    hits_total, exact_total, win_total, n_races = 0, 0, 0, 0
    
    for race_id, group in meta.groupby('raceId'):
        if len(group) < 3:
            continue
        actual_podium = group[group['positionOrder'] <= 3]['driverRef'].tolist()
        actual_p1 = group[group['positionOrder'] == 1]['driverRef'].values
        
        top3 = group.nlargest(3, 'score')
        predicted_podium = top3['driverRef'].tolist()
        predicted_p1 = top3.iloc[0]['driverRef']
        
        hits_total += sum(1 for d in actual_podium if d in predicted_podium)
        exact_total += int(all(d in predicted_podium for d in actual_podium))
        win_total += int(actual_p1[0] == predicted_p1)
        n_races += 1
    
    phr = hits_total / (n_races * 3)
    win_acc = win_total / n_races
    exact = exact_total / n_races
    
    print(f'=== {label} ===')
    print(f'  Podium Hit Rate:  {phr:.4f}')
    print(f'  Winner Accuracy:  {win_acc:.4f}')
    print(f'  Exact Drivers:    {exact:.4f}')
    print()
    
    return {
        'Model': label,
        'Podium Hit Rate': phr,
        'Winner Accuracy': win_acc,
        'Exact Drivers': exact
    }

# Ensemble score
ensemble_score_te = w1_opt * clf_prob_te + w2_opt * ranker_score_te_norm + w3_opt * 0

# Evaluasi semua model
all_results = []
all_results.append(evaluate_on_test(te_meta, clf_prob_te, 'Classifier Only'))
all_results.append(evaluate_on_test(te_meta, ranker_score_te, 'Ranker Only'))
all_results.append(evaluate_on_test(te_meta, ensemble_score_te, 'Ensemble (Clf+Rank)'))

final_comparison = pd.DataFrame(all_results)
print('\n=== FINAL COMPARISON (Test Set 2024-2025) ===')
display(final_comparison)

In [ ]:
# Per-season breakdown
def evaluate_per_season(te_meta, scores, label='Model'):
    meta = te_meta.copy()
    meta['score'] = scores
    
    season_results = []
    for year, group in meta.groupby('year'):
        hits, n_races = 0, 0
        for race_id, race_group in group.groupby('raceId'):
            if len(race_group) < 3:
                continue
            actual = race_group[race_group['positionOrder'] <= 3]['driverRef'].tolist()
            top3 = race_group.nlargest(3, 'score')
            pred = top3['driverRef'].tolist()
            hits += sum(1 for d in actual if d in pred)
            n_races += 1
        
        season_results.append({
            'Model': label,
            'Season': int(year),
            'Podium Hit Rate': hits / (n_races * 3) if n_races > 0 else 0,
            'Races': n_races
        })
    return pd.DataFrame(season_results)

# Gabungkan per-season untuk semua model
season_dfs = []
for label, scores in [
    ('Classifier Only', clf_prob_te),
    ('Ranker Only', ranker_score_te),
    ('Ensemble (Clf+Rank)', ensemble_score_te)
]:
    season_dfs.append(evaluate_per_season(te_meta, scores, label))

season_df = pd.concat(season_dfs, ignore_index=True)
print('=== PERFORMANCE PER SEASON ===')
display(season_df.pivot_table(index='Model', columns='Season', values='Podium Hit Rate', aggfunc='first'))

In [ ]:
# Visualisasi per-season
plt.figure(figsize=(10, 5))
for model in season_df['Model'].unique():
    subset = season_df[season_df['Model'] == model]
    plt.plot(subset['Season'], subset['Podium Hit Rate'],
             marker='o', label=model, linewidth=2)

plt.xlabel('Season')
plt.ylabel('Podium Hit Rate')
plt.title('Model Performance per Season (Test Set)')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/season_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/season_comparison.png')

## 6. Simpan Hasil

In [ ]:
import joblib
import os

os.makedirs('../models/ensemble/', exist_ok=True)
os.makedirs('../reports/metrics/', exist_ok=True)

# Simpan ensemble model
joblib.dump({
    'classifier': clf,
    'ranker': ranker,
    'w1_classifier': w1_opt,
    'w2_ranker': w2_opt,
    'w3_baseline': w3_opt,
}, '../models/ensemble/ensemble_model.pkl')
print('Ensemble model saved: models/ensemble/ensemble_model.pkl')

# Simpan hasil final
final_comparison.to_csv('../reports/metrics/ensemble_results.csv', index=False)
print('Ensemble results saved: reports/metrics/ensemble_results.csv')

wf_df.to_csv('../reports/metrics/walk_forward_results.csv', index=False)
print('Walk-forward results saved: reports/metrics/walk_forward_results.csv')

season_df.to_csv('../reports/metrics/season_performance.csv', index=False)
print('Season performance saved: reports/metrics/season_performance.csv')

In [ ]:
print('=== RINGKASAN FINAL ===')
print('=' * 60)
print(f'{"Model":<25} {"PHR":<12} {"Winner":<12} {"Exact":<12}')
print('-' * 60)
for _, row in final_comparison.iterrows():
    print(f'{row["Model"]:<25} {row["Podium Hit Rate"]:<12.4f} '
          f'{row["Winner Accuracy"]:<12.4f} {row["Exact Drivers"]:<12.4f}')
print('=' * 60)
print(f'Bobot ensemble: Classifier={w1_opt:.1f}, Ranker={w2_opt:.1f}, Baseline={w3_opt:.1f}')